# 4. Infinite-Variance / Heavy-Tail Check via Hill Estimator


## Load the derived residual series

This notebook uses the residual series created by `02_decomposition.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

residual_series = pd.read_csv(
    "data/residual_series.csv",
    parse_dates=["date"],
    index_col="date"
)["residual"].dropna()

print(f"Loaded {len(residual_series)} residual observations.")


## Concepts

A **heavy-tailed** distribution gives relatively more probability to extreme observations than a light-tailed distribution. The **Hill estimator** is a tail-index estimator used to study extreme-value behavior.

### Analogy
If one basket contains unusually large stones much more often than another basket, it gives an intuition for heavier tails.


In [ ]:
import seaborn as sns
from scipy.stats import norm
import matplotlib.pyplot as plt
import numpy as np

# 1. Boxplot to check for heavy tails
plt.figure(figsize=(8, 4))
sns.boxplot(x=residual_series, color='salmon')
plt.title('Boxplot of Final Residuals')
plt.xlabel('Residual Value')
plt.grid(True)
plt.tight_layout()
plt.show()

# 2. Hill estimator function
def hill_estimator(data, k_values):
    data = np.sort(np.abs(data))[::-1]
    hill_estimates = []
    for k in k_values:
        if k < len(data):
            top_k = data[:k]
            hill = (1 / k) * np.sum(np.log(top_k / data[k]))
            alpha_hat = 1 / hill
            hill_estimates.append(alpha_hat)
        else:
            hill_estimates.append(np.nan)
    return hill_estimates

# Define range of k and compute estimates
k_vals = range(10, min(500, len(residual_series) - 1))
hill_alpha = hill_estimator(residual_series.values, k_vals)

# Choose a stable region for alpha averaging (adjust if needed)
stable_start = 100
stable_end = 200
stable_k_indices = [i for i in range(stable_start, stable_end) if i - 10 < len(hill_alpha)]
stable_alphas = [hill_alpha[i - 10] for i in stable_k_indices]
alpha_mean = np.mean(stable_alphas)

# Plot Hill estimator with horizontal line
plt.figure(figsize=(10, 4))
plt.plot(k_vals, hill_alpha, color='blue', label='Estimated α')
plt.axhline(y=alpha_mean, color='red', linestyle='--', label=f'Mean α ≈ {alpha_mean:.2f} (k={stable_start}-{stable_end})')
plt.title('Hill Plot: Estimated Alpha vs k')
plt.xlabel('k (Top Order Statistics)')
plt.ylabel('Estimated Alpha')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
